<a href="https://colab.research.google.com/github/Kazi-Sadman/Drug-solubility-prediction/blob/main/Thesis_new_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
import torch

if torch.cuda.is_available():
    print("✅ GPU CONNECTED")
    print("GPU name :", torch.cuda.get_device_name(0))
    print("Memory   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("❌ GPU NOT CONNECTED — CPU চলছে")

✅ GPU CONNECTED
GPU name : Tesla T4
Memory   : 15.64 GB


# =============================================================================
# THESIS: An Explainable Language-Aware Cross-Lingual Framework for
#         Bangla-English Fake News Detection Using Multilingual Transformers
#         and Classical Machine Learning
#
# FILE: Single-file end-to-end implementation for Google Colab / Jupyter
# SEED: 42  |  TEST SET: frozen until Section 26 (Final Evaluation)
# ===================================================


# =============================================================================
# SECTION 1 — ENVIRONMENT SETUP
# =============================================================================
# Installs all dependencies needed for the full pipeline.
# Safe to re-run (pip skips already-installed packages).

In [1]:

!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn \
    transformers datasets accelerate sentencepiece langdetect lime \
    joblib tqdm pyyaml openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 30.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 19.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


# SECTION 2 — IMPORTS
# =============================================================================
# Import every library used later. Names are grouped by purpose.

# --- Standard library ---

In [2]:
# =============================================================================
# THESIS: An Explainable Language-Aware Cross-Lingual Framework for
#         Bangla-English Fake News Detection Using Multilingual Transformers
#         and Classical Machine Learning
#
# FILE: Single-file end-to-end implementation for Google Colab / Jupyter
# SEED: 42  |  TEST SET: frozen until Section 26 (Final Evaluation)
# =============================================================================


# =============================================================================
# SECTION 1 — ENVIRONMENT SETUP
# =============================================================================
# Installs all dependencies needed for the full pipeline.
# Safe to re-run (pip skips already-installed packages).

!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn \
    transformers datasets accelerate sentencepiece langdetect lime \
    joblib tqdm pyyaml openpyxl


# =============================================================================
# SECTION 2 — IMPORTS
# =============================================================================
# Import every library used later. Names are grouped by purpose.

# --- Standard library ---
import os
import re
import sys
import json
import time
import html
import random
import unicodedata
import warnings
from pathlib import Path
from collections import Counter

# --- Data handling ---
import numpy as np
import pandas as pd

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Classical machine learning ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_fscore_support
)
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBClassifier

# --- Language identification ---
from langdetect import detect as detect_language_langdetect
from langdetect import DetectorFactory, LangDetectException
DetectorFactory.seed = 0

# --- Explainability ---
from lime.lime_text import LimeTextExplainer

# --- Deep learning (transformers) ---
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- Persistence ---
import joblib

# --- Notebook friendliness ---
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

In [3]:
# =============================================================================
# SECTION 3 — CONFIGURATION AND RANDOM SEEDS (Reproducibility)
# =============================================================================

# -------- 3.1 Random seed (reproducibility) --------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# -------- 3.2 Dataset schema (from dataset audit) --------
COLUMN_NEWS_ID       = "id"
COLUMN_TITLE         = "title"
COLUMN_NEWS_BODY     = "news_details"
COLUMN_LANGUAGE      = "language"
COLUMN_CATEGORY      = "category"
COLUMN_LABEL         = "label"

LABEL_MAPPING        = {"Real": 0, "Fake": 1}
LABEL_NAMES          = ["Real", "Fake"]
NUMBER_OF_CLASSES    = 2

# Columns never used as model features (leakage / metadata risk)
METADATA_COLUMNS     = [COLUMN_NEWS_ID, COLUMN_CATEGORY]

# -------- 3.3 Output directory layout --------
OUTPUT_ROOT_DIR      = "outputs"
FIGURES_DIR          = f"{OUTPUT_ROOT_DIR}/figures"
REPORTS_DIR          = f"{OUTPUT_ROOT_DIR}/reports"
MODELS_DIR           = f"{OUTPUT_ROOT_DIR}/models"
PREDICTIONS_DIR      = f"{OUTPUT_ROOT_DIR}/predictions"
LIME_EXPLANATIONS_DIR= f"{OUTPUT_ROOT_DIR}/lime"

for output_directory in [
    OUTPUT_ROOT_DIR, FIGURES_DIR, REPORTS_DIR,
    MODELS_DIR, PREDICTIONS_DIR, LIME_EXPLANATIONS_DIR
]:
    Path(output_directory).mkdir(parents=True, exist_ok=True)

# -------- 3.4 Split ratios (Section 8) --------
TRAIN_FRACTION       = 0.70
VALIDATION_FRACTION  = 0.15
TEST_FRACTION        = 0.15

# -------- 3.5 TF-IDF hyperparameters (Section 11) --------
TFIDF_MIN_DOCUMENT_FREQUENCY = 2
TFIDF_MAX_DOCUMENT_FREQUENCY = 0.95
TFIDF_USE_SUBLINEAR_TF       = True
TFIDF_MAX_NUMBER_OF_FEATURES = 120_000

# N-gram configurations to evaluate during TF-IDF ablation
TFIDF_NGRAM_CANDIDATES = [(1, 1), (1, 2)]

# -------- 3.6 Classical model hyperparameter grids (Section 11) --------
SVM_REGULARIZATION_GRID       = [0.1, 1.0, 10.0]
LOGREG_REGULARIZATION_GRID    = [0.1, 1.0, 10.0]
NAIVE_BAYES_ALPHA_GRID        = [0.1, 0.5, 1.0]

XGBOOST_FIXED_PARAMETERS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.1,
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_jobs": -1,
    "use_label_encoder": False,
}

# -------- 3.7 Cross-validation (Section 9) --------
NUMBER_OF_CROSS_VALIDATION_FOLDS = 5

# -------- 3.8 Transformer training (Section 14) --------
MULTILINGUAL_TRANSFORMER_NAMES = [
    "bert-base-multilingual-cased",   # mBERT
    "xlm-roberta-base",               # XLM-R
]
TRANSFORMER_MAX_SEQUENCE_LENGTH = 256
TRANSFORMER_BATCH_SIZE          = 16
TRANSFORMER_NUMBER_OF_EPOCHS    = 3
TRANSFORMER_LEARNING_RATE       = 2e-5
TRANSFORMER_WEIGHT_DECAY        = 0.01
TRANSFORMER_WARMUP_RATIO        = 0.1
TRANSFORMER_EARLY_STOP_PATIENCE = 1

# -------- 3.9 Leakage detection (Section 5 / 9) --------
NEAR_DUPLICATE_COSINE_THRESHOLD = 0.90

# -------- 3.10 LIME (Section 25) --------
LIME_NUMBER_OF_FEATURES = 15
LIME_NUMBER_OF_SAMPLES  = 1000

# -------- 3.11 Device --------
COMPUTE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[INFO] Random seed set to {RANDOM_SEED}")
print(f"[INFO] Compute device: {COMPUTE_DEVICE}")
if COMPUTE_DEVICE == "cpu":
    print("[WARN] No GPU detected. Transformer fine-tuning will be SKIPPED.")


# =============================================================================
# Utility functions used throughout the notebook
# =============================================================================

def write_json_report(report_object, output_file_path):
    """
    Persist a Python object (dict/list) as a JSON file using UTF-8.
    Ensures Bangla characters are preserved.
    """
    Path(output_file_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_file_path, "w", encoding="utf-8") as json_file:
        json.dump(report_object, json_file, ensure_ascii=False,
                  indent=2, default=str)


def print_section_banner(section_number, section_title):
    """Print a clear section banner for readability in the notebook."""
    banner = "=" * 78
    print(f"\n{banner}\nSECTION {section_number} — {section_title}\n{banner}")



[INFO] Random seed set to 42
[INFO] Compute device: cuda


In [ ]:
# =============================================================================
# SECTION 4 — DATASET LOADING
# =============================================================================
print_section_banner(4, "DATASET LOADING")

# In Google Colab, this cell prompts you to upload the CSV file.
# In Jupyter, place the CSV in the same folder as the notebook.
try:
    from google.colab import files as colab_files
    print("Please upload your dataset CSV (e.g., ben_fnd_16k (1).csv):")
    uploaded_files = colab_files.upload()
    RAW_DATASET_FILEPATH = list(uploaded_files.keys())[0]
    print(f"[INFO] Uploaded file: {RAW_DATASET_FILEPATH}")
except ImportError:
    RAW_DATASET_FILEPATH = "ben_fnd_16k (1).csv"
    print(f"[INFO] Not in Colab. Expecting local file: {RAW_DATASET_FILEPATH}")

# Fallback: try common alternative filenames
if not Path(RAW_DATASET_FILEPATH).exists():
    for alternative_filename in ["ben_fnd_16k.csv", "data.csv", "dataset.csv"]:
        if Path(alternative_filename).exists():
            RAW_DATASET_FILEPATH = alternative_filename
            break

assert Path(RAW_DATASET_FILEPATH).exists(), (
    f"Dataset not found at {RAW_DATASET_FILEPATH}. "
    "Please upload it or set RAW_DATASET_FILEPATH manually."
)

# Load with Python engine to tolerate malformed rows (audit detected one).
raw_dataset_dataframe = pd.read_csv(
    RAW_DATASET_FILEPATH,
    engine="python",
    on_bad_lines="skip",
)
raw_dataset_dataframe.columns = [column.strip() for column in raw_dataset_dataframe.columns]

print(f"[INFO] Raw dataset shape: {raw_dataset_dataframe.shape}")
print(f"[INFO] Columns: {list(raw_dataset_dataframe.columns)}")
raw_dataset_dataframe.head(3)




SECTION 4 — DATASET LOADING
Please upload your dataset CSV (e.g., ben_fnd_16k (1).csv):
